<div style="text-align: left;font-size: 12px">For questions or problems regarding the notebook don't hesitate to ask <a href=mailto:matthias.krauss@fu-berlin.de>Matthias Krauss</a></div>

---

<table style="width: 100%">
    <tr>
    <td style="width: 5% text-align:center">
    </td>
    <td style="width: 18% text-align:center">
        <h3><ins>Exercise 3a: Krotov 2-Level</ins></h3>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_3b.ipynb"><h3>Exercise 3b: Krotov 3-Level</h3></a>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_3c.ipynb"><h3>Exercise 3c: Krotov 3-Level (chiral)</h3></a>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_3d.ipynb"><h3>Exercise 3d: Krotov 4-Level (chiral)</h3></a>
    </td>
    <td style="width: 5% text-align:center">
        <a href="exercise_3b.ipynb"><h3>$\rightarrow$</h3></a>
    </td>
    </tr>
</table>

---

First, we need to load some of the libraries that we will need throughout this notebook. Click on the cell and hit **SHIFT + ENTER** to evaluate it.

In [ ]:
import krotov  # package for optimizing with krotov
import matplotlib  # package for plotting
import matplotlib.pylab as plt
import numpy as np  # package for numerical functions such as cos, sin, etc.
import qutip  # QUantum Toolbox In Python
import scipy

# Some functions for easy access:
from numpy import pi, sqrt, exp, sin, cos

We are using some Bloch sphere visualizations in this tutorial that benefit from being interactive, so we will activate an interactive backend for `matplotlib`:

In [ ]:
%matplotlib widget

Note that this requires the `ipympl` package to be installed in the same version both in the project environment and in the environment providing the Jupyter application. If you are having trouble with the plots in this notebook, delete the above cell, restart the kernel, and rerun the notebook. The plots won't be interactive, but you'll still be able to follow along this tutorial.

# Exercise 4 -  State-to-state transfer in a two-level system using Krotov

$\newcommand{tr}[0]{\operatorname{tr}}
\newcommand{diag}[0]{\operatorname{diag}}
\newcommand{abs}[0]{\operatorname{abs}}
\newcommand{pop}[0]{\operatorname{pop}}
\newcommand{aux}[0]{\text{aux}}
\newcommand{opt}[0]{\text{opt}}
\newcommand{tgt}[0]{\text{tgt}}
\newcommand{init}[0]{\text{init}}
\newcommand{lab}[0]{\text{lab}}
\newcommand{rwa}[0]{\text{rwa}}
\newcommand{bra}[1]{\langle#1\vert}
\newcommand{ket}[1]{\vert#1\rangle}
\newcommand{Bra}[1]{\left\langle#1\right\vert}
\newcommand{Ket}[1]{\left\vert#1\right\rangle}
\newcommand{Braket}[2]{\left\langle #1\vphantom{#2} \mid
#2\vphantom{#1}\right\rangle}
\newcommand{op}[1]{\hat{#1}}
\newcommand{Op}[1]{\hat{#1}}
\newcommand{dd}[0]{\,\text{d}}
\newcommand{Liouville}[0]{\mathcal{L}}
\newcommand{DynMap}[0]{\mathcal{E}}
\newcommand{identity}[0]{\mathbf{1}}
\newcommand{Norm}[1]{\lVert#1\rVert}
\newcommand{Abs}[1]{\left\vert#1\right\vert}
\newcommand{avg}[1]{\langle#1\rangle}
\newcommand{Avg}[1]{\left\langle#1\right\rangle}
\newcommand{AbsSq}[1]{\left\vert#1\right\vert^2}
\newcommand{Re}[0]{\operatorname{Re}}
\newcommand{Im}[0]{\operatorname{Im}}$

## Setup

### Define the Hamiltonian

In the
following the Hamiltonian, guess field and
states are defined.

The Hamiltonian
$\op{H}_{0} = - \omega \op{\sigma}_{z}$
represents a
simple qubit with energy
level splitting $\omega$ in the basis
$\{\ket{0},\ket{1}\}$. The control
field
$\epsilon(t)$ is assumed to couple via
the
Hamiltonian $\op{H}_{1}(t) =
\epsilon(t) \op{\sigma}_{x}$ to the qubit,
i.e., the control
field effectively
drives
transitions between both qubit
states.

In [ ]:
def ham_and_states(omega=1.0, eps0=(lambda t, args: 1.0)):
    """Two-level-system Hamiltonian

    Args:
        omega (float): energy separation of the qubit levels
        eps0 (func): The driving field eps0(t, args)
    """
    H0 = -0.5 * omega * qutip.operators.sigmaz()
    H1 = qutip.operators.sigmax()

    psi0 = qutip.Qobj(np.array([1, 0]))  # State |0⟩
    psi1 = qutip.Qobj(np.array([0, 1]))  # State |1⟩

    return ([H0, [H1, eps0]], psi0, psi1)

In addition, we define a shape function $S(t)$ which takes care of
experimental limits such as the necessity of finite ramps
at the beginning and
end of the control field.

In [ ]:
def S(t):
    """Shape function for the initial and field update"""
    return krotov.shapes.flattop(
        t, t_start=0, t_stop=10, t_rise=0.5, t_fall=0.5, func="sinsq"
    )

This $S(t)$ will be used in two contexts: in the optimization with Krotov's method later on in this tutorial, it will shape the pulse update, ensuring that the boundary conditions are maintained in every iteration of the optimization.

Before that, we will also use $S(t)$ to multiply the `eps0` when calling `ham_as_states`:

In [ ]:
def shape_field(eps0):
    """Applies the shape function S(t) to the guess field"""
    eps0_shaped = lambda t, args: eps0(t, args) * S(t)
    return eps0_shaped

This allows playing around with different functions for `eps0` that may or may not have suitable boundary conditions.

### Simulate dynamics of the guess field

Before heading towards the optimization
procedure, we first simulate the
dynamics under the guess field
$\epsilon_{0}(t)$.
The following plot shows the guess field $\epsilon_{0}(t)$.

However, before we can propagate the state under the guess field, we need to define the time grid of the
dynamics. As an example, we define the
initial state to be at time $t=0$ and
consider a total propagation time of
$T=4$. The entire time grid is divided into
$n_{t}=80$ equidistant time steps (so, 81 time grid points).

In [ ]:
tlist = np.linspace(0, 10, 81)

And we of course also have to define the guess pulse itself.

In [ ]:
def guess_pulse(t, args):
    A = .1
    σ = 2
    E = A * exp(-((t - 5) ** 2) / (2 * σ ** 2)) * cos(3*t)
    return E

In the Hamiltonian, we multiply `guess_pulse` with $S(t)$ via `shape_field`:

In [ ]:
H, psi0, psi1 = ham_and_states(eps0=shape_field(guess_pulse))

Feel free to play around with `guess_pulse` and the construction of the Hamiltonian.

The total field looks as follows:

In [ ]:
def plot_pulse(pulse, tlist):
    fig, ax = plt.subplots(figsize=(8, 5))
    if callable(pulse):
        pulse = np.array([pulse(t, args=None) for t in tlist])
    ax.plot(tlist, pulse)
    ax.set_xlabel("time")
    ax.set_ylabel("pulse amplitude")
    plt.show(fig)

plt.close("all")
plot_pulse(H[1][1], tlist)

Then, we solve the equation of motion for the initial state $\ket{\Psi_{\init}}=\ket{0}$ and the Hamiltonian $\op{H}(t)$
defining its evolution. To this end, we define the projectors $\op{P}_0 = \ket{0}\bra{0}$ and $\op{P}_1 = \ket{1}\bra{1}$ to compute their expecatation values. Afterwards, we plot the dynamics.

In [ ]:
proj0 = psi0 * psi0.dag()
proj1 = psi1 * psi1.dag()

guess_dynamics = qutip.mesolve(H, psi0, tlist, e_ops=[proj0, proj1])


def plot_population(result, ylim=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(result.times, result.expect[0], label="0")
    ax.plot(result.times, result.expect[1], label="1")
    ax.legend()
    ax.set_xlabel("time")
    ax.set_ylabel("population")
    if ylim is not None:
        ax.set_ylim(ylim)
    plt.show(fig)

plt.close("all")
plot_population(guess_dynamics)

Note that there is a small amount of oscillation in the dynamics, which we can see by zooming in the dynamics of the $\ket{1}$ state

In [ ]:
plt.close("all")
plot_population(guess_dynamics, ylim=(0, 0.0016))

We can do the same again, but plot the trajectory on the Bloch sphere instead. Try to understand how the mesolve routine is different compared to the one for plotting the population.

In [ ]:
σ_x = qutip.sigmax()
σ_y = qutip.sigmay()
σ_z = qutip.sigmaz()

guess_dynamics = qutip.mesolve(H, psi0, tlist, e_ops=[σ_x, σ_y, σ_z])


def plot_bloch(result):
    b = qutip.Bloch(view=[-60, 30])
    exp_x = result.expect[0]
    exp_y = result.expect[1]
    exp_z = result.expect[2]
    b.point_color = plt.get_cmap("viridis_r")(tlist / tlist[-1])  # set nice colormap
    b.add_points([exp_x, exp_y, exp_z], "m")
    b.frame_alpha = 0.1
    b.make_sphere()
    plt.show()

plt.close("all")
plot_bloch(guess_dynamics)

## Optimization

### Define the optimization target

We want to optimize a simple state-to-state
transfer
from initial state $\ket{\Psi_{\init}} = \ket{0}$ to the target state
$\ket{\Psi_{\tgt}} = \ket{1}$, which we want to reach at final time $T$. Note
that we also have to pass the Hamiltonian $\op{H}(t)$ that determines the
dynamics of
the system.

From a mathematical perspective we optimize the guess field $\epsilon_{0}(t)$ such
that the intended state-to-state transfer $\ket{\Psi_{\init}} \rightarrow
\ket{\Psi_{\tgt}}$ is solved.
To this end, we
choose the functional to be $F = F_{ss}$ with
\begin{equation}
F_{ss}
=
\left|\Braket{\Psi(T)}{\Psi_{\tgt}}\right|
\end{equation}

with
$\ket{\Psi(T)}$ the
forward propagated state of $\ket{\Psi_{\init}}$. Maximizing $F_{ss}$ is equivalent to minimizing

In [ ]:
? krotov.functionals.J_T_ss

The functional is evaluated based on the initial state forward-propagated under a specific time-dependent Hamiltonian $\Op{H}(t)$ and the associated target state. The initial state, Hamiltonian, and target state are collected in an `Objective` object that will be passed to the `J_T_ss` function.

In [ ]:
objectives = [krotov.Objective(initial_state=psi0, target=psi1, H=H)]

The *result* of the propagation is available to `J_T_ss` as `fw_states_T`.

### Using Krotov's method

In an optimization with Krotov's method, we have the option of using $S(t)$ as the "update shape" for
$\epsilon_0(t)$. Wherever $S(t)$ is zero, the optimization will not change the
value of the control from the original guess. In general, this shape function can be different from the one used to shape the guess pulse. In addition, we have to choose `lambda_a` for each control
field. It controls the update magnitude of the respective field in each iteration. These options are collected in a dictionary of `pulse_options`:

In [ ]:
pulse_options = {H[1][1]: dict(lambda_a=25, update_shape=S)}

We can now collect everything into a call to `optimize_pulse`:

In [ ]:
oct_result = krotov.optimize_pulses(
    objectives,
    pulse_options=pulse_options,
    tlist=tlist,
    propagator=krotov.propagators.expm,
    chi_constructor=krotov.functionals.chis_ss,
    info_hook=krotov.info_hooks.print_table(
        J_T=krotov.functionals.J_T_ss,
        show_g_a_int_per_pulse=False,
        unicode=False,
    ),
    check_convergence=krotov.convergence.Or(
        krotov.convergence.value_below(1e-3, name="J_T"),
        krotov.convergence.check_monotonic_error,
    ),
    iter_stop=50,
)

### Simulate dynamics of the optimized field

Having obtained the optimized
control field, we can now
plot it and calculate the
population dynamics under
this field.

In [ ]:
plt.close("all")
plot_pulse(oct_result.optimized_controls[0], tlist)

opt_dynamics = oct_result.optimized_objectives[0].mesolve(tlist, e_ops=[proj0, proj1])
plot_population(opt_dynamics)

opt_dynamics = oct_result.optimized_objectives[0].mesolve(tlist, e_ops=[σ_x, σ_y, σ_z])
plot_bloch(opt_dynamics)

## Tasks

1) Vary the numerical parameters `lambda_a` and $n_{t}$ to improve the optimization. You should be able to reach the desired fidelity of 99% within less than 50 iterations.

2) Try to improve the guess pulse to converge faster. Hint: The interesting parameters are `ampl0` and $T$/`t_stop` (Keep in mind to change it in the shape $S$ and in the time grid `tlist`). Also, a constant pulse might not be the best option as a guess pulse.

---

<table style="width: 100%">
    <tr>
    <td style="width: 5% text-align:center">
    </td>
    <td style="width: 18% text-align:center">
        <h3><ins>Exercise 3a: Krotov 2-Level</ins></h3>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_3b.ipynb"><h3>Exercise 3b: Krotov 3-Level</h3></a>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_3c.ipynb"><h3>Exercise 3c: Krotov 3-Level (chiral)</h3></a>
    </td>
    <td style="width: 18% text-align:center">
        <a href="exercise_3d.ipynb"><h3>Exercise 3d: Krotov 4-Level (chiral)</h3></a>
    </td>
    <td style="width: 5% text-align:center">
        <a href="exercise_3b.ipynb"><h3>$\rightarrow$</h3></a>
    </td>
    </tr>
</table>

---